In [2]:
# =============================================================================
# CELL 1 — GP Reservation Merge: Rajasthan 2005 and 2010 Elections
# =============================================================================
#
# Purpose:
#   Merge GP-level election reservation status for Rajasthan (2005 and 2010
#   election cycles) onto the cluster-to-GP crosswalk built in
#   cluster_GPs_cleaning.ipynb. The merged dataset will allow us to:
#     (1) Identify the reservation status of each cluster's assigned GP
#     (2) Check whether at-risk clusters have adjacent GPs with the same
#         reservation status (i.e., whether spatial ambiguity actually
#         affects treatment assignment)
#     (3) Extend the Monte Carlo simulation to compute P(correct treatment
#         status) rather than just P(primary GP)
#
# Input files:
#   - sp_2005_2010_manually_reviewed.csv  — GP reservation status for
#     Rajasthan 2005 and 2010 elections
#   - nfhs4_cluster_gp_crosswalk.csv      — NFHS4 cluster-to-GP crosswalk
#   - nfhs5_cluster_gp_crosswalk.csv      — NFHS5 cluster-to-GP crosswalk
#
# Note on Rajasthan delimitation:
#   GP boundaries were redrawn in Rajasthan in 2015. NFHS4 (2015-16)
#   reflects the pre-delimitation GP structure relevant for 2005 and 2010
#   elections. NFHS5 (2019-21) reflects the post-delimitation structure.
#   This notebook focuses on NFHS4 as the primary analysis round.

In [3]:
# =============================================================================
# CELL 2 — Imports and Paths
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ── Relative paths ────────────────────────────────────────────────────────────
DATA_DIR    = Path("../data")
OUTPUT_DIR  = Path("../outputs")

RESERVATION_PATH  = DATA_DIR / "rajasthan gp reservations" / "sp_2005_2010_manually_reviewed.csv"
NFHS4_CROSSWALK   = DATA_DIR / "nfhs4_cluster_gp_crosswalk.csv"
NFHS5_CROSSWALK   = DATA_DIR / "nfhs5_cluster_gp_crosswalk.csv"

# ── Verify files exist ────────────────────────────────────────────────────────
print("Checking required files:")
for path in [RESERVATION_PATH, NFHS4_CROSSWALK, NFHS5_CROSSWALK]:
    status = "✓" if path.exists() else "✗ NOT FOUND"
    print(f"  {status}  {path}")

Checking required files:
  ✓  ../data/rajasthan gp reservations/sp_2005_2010_manually_reviewed.csv
  ✓  ../data/nfhs4_cluster_gp_crosswalk.csv
  ✓  ../data/nfhs5_cluster_gp_crosswalk.csv


In [4]:
# =============================================================================
# CELL 3 — Load and Inspect Reservation Data
# =============================================================================
# Before attempting any merge, inspect the reservation data carefully to
# understand the join key and data structure.

reservations = pd.read_csv(RESERVATION_PATH)

print(f"Rows: {len(reservations)}")
print(f"Columns: {reservations.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(reservations.head())

print(f"\n── Key field inspection ──")
print(f"\nkey_2005 sample values:")
print(reservations["key_2005"].dropna().head(10).tolist())

print(f"\nkey_2010 sample values:")
print(reservations["key_2010"].dropna().head(10).tolist())

print(f"\nreservation_2005 unique values:")
print(reservations["reservation_2005"].value_counts())

print(f"\nreservation_2010 unique values:")
print(reservations["reservation_2010"].value_counts())

print(f"\nnuke unique values:")
print(reservations["nuke"].value_counts())

print(f"\ngp_new_2005 sample values:")
print(reservations["gp_new_2005"].dropna().head(10).tolist())

print(f"\nUnique districts in reservation data:")
print(reservations["dist_name_2005"].value_counts())

Rows: 9860
Columns: ['sl_no_2005', 'dist_name_2005', 'samiti_name_2005', 'gp_2005', 'reservation_2005', 'name_2005', 'sex_2005', 'category_2005', 'gp_new_2005', 'dist_name_new_2005', 'samiti_name_new_2005', 'dist_2010_2005', 'key_2005', 'sl_no_2010', 'dist_name_2010', 'samiti_name_2010', 'gp_2010', 'reservation_2010', 'name_2010', 'sex_2010', 'category_2010', 'gp_new_2010', 'dist_name_new_2010', 'samiti_name_new_2010', 'key_2010', 'dist', 'nuke']

First 5 rows:
   sl_no_2005 dist_name_2005 samiti_name_2005      gp_2005 reservation_2005  \
0           2          AJMER            ARAIN       AJGARA               SC   
1           1          AJMER            ARAIN     AAKODIYA              OBC   
2           3          AJMER            ARAIN        ARAIN               ST   
3           4          AJMER            ARAIN  BHAGWANPURA              GEN   
4           5          AJMER            ARAIN     BHAMOLAV              GEN   

     name_2005 sex_2005 category_2005  gp_new_2005 dist_nam

In [5]:
# =============================================================================
# CELL 4 — Clean Reservation Data and Create Treatment Variables
# =============================================================================

# Drop rows flagged by nuke field (problematic records from manual review)
print(f"Rows before dropping nuke: {len(reservations)}")
reservations_clean = reservations[reservations["nuke"].isna()].copy()
print(f"Rows after dropping nuke:  {len(reservations_clean)}")
print(f"Dropped:                   {len(reservations) - len(reservations_clean)}")

# Create binary treatment variables
# Women-reserved = any category with W suffix
women_cats_2005 = ["GEN W", "ST W", "SC W", "OBC W"]
women_cats_2010 = ["GENW", "STW", "SCW", "OBCW"]

reservations_clean["reserved_women_2005"] = (
    reservations_clean["reservation_2005"].isin(women_cats_2005)
).astype(int)

reservations_clean["reserved_women_2010"] = (
    reservations_clean["reservation_2010"].isin(women_cats_2010)
).astype(int)

print(f"\nGPs reserved for women:")
print(f"  2005: {reservations_clean['reserved_women_2005'].sum()} "
      f"({reservations_clean['reserved_women_2005'].mean():.1%})")
print(f"  2010: {reservations_clean['reserved_women_2010'].sum()} "
      f"({reservations_clean['reserved_women_2010'].mean():.1%})")

# How many GPs were reserved in both cycles?
reservations_clean["reserved_both"] = (
    (reservations_clean["reserved_women_2005"] == 1) &
    (reservations_clean["reserved_women_2010"] == 1)
).astype(int)

print(f"  Both 2005 and 2010: {reservations_clean['reserved_both'].sum()} "
      f"({reservations_clean['reserved_both'].mean():.1%})")

Rows before dropping nuke: 9860
Rows after dropping nuke:  8290
Dropped:                   1570

GPs reserved for women:
  2005: 2770 (33.4%)
  2010: 3920 (47.3%)
  Both 2005 and 2010: 1312 (15.8%)


In [7]:
# =============================================================================
# CELL 5 — Load LGD Village-to-GP File and Build Match Key
# =============================================================================

VILLAGE_TO_GP_PATH = DATA_DIR / "All India Village to GP LGD codes.csv"

village_to_gp = pd.read_csv(VILLAGE_TO_GP_PATH, encoding="latin-1")

# Filter to Rajasthan
village_to_gp_rj = village_to_gp[
    village_to_gp["State Code"].astype(str).str.strip().isin(["8", "08"])
].copy()

print(f"Rajasthan rows in village-to-GP file: {len(village_to_gp_rj)}")
print(f"Columns: {village_to_gp_rj.columns.tolist()}")
print(f"\nSample rows:")
print(village_to_gp_rj[["District Name", "Subdistrict Name",
                          "Gram Panchayat Name",
                          "Gram Panchayat LGD Code"]].head(10))

# Build one row per unique GP
gp_lookup = village_to_gp_rj.groupby("Gram Panchayat LGD Code").first().reset_index()[[
    "Gram Panchayat LGD Code",
    "Gram Panchayat Name",
    "District Name",
    "Subdistrict Name"
]].copy()

gp_lookup.columns = ["gp_lgd_code", "gp_name_lgd",
                      "district_lgd", "subdistrict_lgd"]

print(f"\nUnique GPs in LGD file (Rajasthan): {len(gp_lookup)}")
print(gp_lookup.head(10))

Rajasthan rows in village-to-GP file: 47971
Columns: ['State Name', 'State Code', 'District Name', 'District Census 2011 Code', 'Subdistrict Name', 'Subdistrict Census 2011 Code', 'Village Name', 'Village Census 2011 Code', 'Village Census 2001 Code', 'Gram Panchayat LGD Code', 'Gram Panchayat Name']

Sample rows:
       District Name Subdistrict Name Gram Panchayat Name  \
426080         Ajmer            Ajmer            Somalpur   
426081         Ajmer            Ajmer               Gegal   
426082         Ajmer            Ajmer             Ajaysar   
426083         Ajmer            Ajmer                 NaN   
426084         Ajmer            Ajmer             Doomara   
426085         Ajmer            Ajmer             Doomara   
426086         Ajmer            Ajmer              Aradka   
426087         Ajmer            Ajmer           Babayacha   
426088         Ajmer            Ajmer            Sedariya   
426089         Ajmer            Ajmer             Mayapur   

        Gram

In [8]:
# =============================================================================
# CELL 6 — Normalize Names and Attempt Match on GP Name + District
# =============================================================================
# Direct key construction fails due to:
#   1. Subdistrict name abbreviations (ARAI vs ARAIN)
#   2. GP name spelling variations (akodiya vs Aakodiya)
#   3. District boundary changes (e.g. Kekri carved from Ajmer in 2022)
#
# Strategy:
#   Step 1: Normalize names (lowercase, strip whitespace, remove punctuation)
#   Step 2: Exact match on normalized GP name + normalized district name
#   Step 3: Fuzzy match for remaining non-matches
#   Step 4: Manual review of low-confidence matches

import re

def normalize_name(s):
    """Lowercase, strip whitespace, remove punctuation and extra spaces."""
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s]", "", s)  # remove punctuation
    s = re.sub(r"\s+", "", s)           # remove all spaces
    return s

# Normalize LGD GP names and district names
gp_lookup["gp_norm"]       = gp_lookup["gp_name_lgd"].apply(normalize_name)
gp_lookup["district_norm"] = gp_lookup["district_lgd"].apply(normalize_name)

# Normalize reservation GP names and district names
reservations_clean["gp_norm_2010"]       = reservations_clean["gp_new_2010"].apply(normalize_name)
reservations_clean["district_norm_2010"] = reservations_clean["dist_name_new_2010"].apply(normalize_name)
reservations_clean["gp_norm_2005"]       = reservations_clean["gp_new_2005"].apply(normalize_name)
reservations_clean["district_norm_2005"] = reservations_clean["dist_name_new_2005"].apply(normalize_name)

print("Sample normalized LGD GP names:")
print(gp_lookup[["gp_name_lgd", "gp_norm", "district_lgd", "district_norm"]].head(10))

print("\nSample normalized reservation GP names:")
print(reservations_clean[["gp_new_2010", "gp_norm_2010",
                           "dist_name_new_2010", "district_norm_2010"]].head(10))

# ── Step 1: Exact match on normalized GP name + district ─────────────────────
merged_exact = reservations_clean.merge(
    gp_lookup[["gp_lgd_code", "gp_name_lgd", "gp_norm",
               "district_lgd", "district_norm", "subdistrict_lgd"]],
    left_on=["gp_norm_2010", "district_norm_2010"],
    right_on=["gp_norm", "district_norm"],
    how="left"
)

matched   = merged_exact["gp_lgd_code"].notna().sum()
unmatched = merged_exact["gp_lgd_code"].isna().sum()
total     = len(merged_exact)

print(f"\n── Exact match results (normalized GP name + district) ──")
print(f"Total GPs:   {total}")
print(f"Matched:     {matched} ({matched/total:.1%})")
print(f"Unmatched:   {unmatched} ({unmatched/total:.1%})")

# Check for duplicate matches (same GP name in same district = ambiguous)
dupes = merged_exact.groupby(["gp_norm_2010", "district_norm_2010"])["gp_lgd_code"].nunique()
ambiguous = (dupes > 1).sum()
print(f"Ambiguous matches (same GP+district → multiple LGD codes): {ambiguous}")

# Sample unmatched rows
print(f"\nSample unmatched reservation GPs:")
print(merged_exact[merged_exact["gp_lgd_code"].isna()][
    ["dist_name_new_2010", "gp_new_2010", "gp_norm_2010", "district_norm_2010"]
].head(15))

Sample normalized LGD GP names:
   gp_name_lgd      gp_norm district_lgd district_norm
0       Ajgara       ajgara        Kekri         kekri
1     Aakodiya     aakodiya        Ajmer         ajmer
2        Arain        arain        Ajmer         ajmer
3  Bhagwanpura  bhagwanpura        Kekri         kekri
4     Bhamolav     bhamolav        Ajmer         ajmer
5    Bhogadeet    bhogadeet        Ajmer         ajmer
6        Birla        birla        Kekri         kekri
7       Borada       borada        Kekri         kekri
8       Dadiya       dadiya        Ajmer         ajmer
9      Deopuri      deopuri        Ajmer         ajmer

Sample normalized reservation GP names:
   gp_new_2010 gp_norm_2010 dist_name_new_2010 district_norm_2010
0       ajgara       ajgara              AJMER              ajmer
1      akodiya      akodiya              AJMER              ajmer
2         arai         arai              AJMER              ajmer
3  bhagwanpura  bhagwanpura              AJMER            

In [9]:
# =============================================================================
# CELL 7 — Match on GP Name Only (Bypass District Boundary Issue)
# =============================================================================
# The LGD file uses 2022 district boundaries. The reservation data uses
# 2005/2010 boundaries. Rajasthan added several new districts in 2022
# (Kekri, Shahpura, Neem Ka Thana, Gangapur City, etc.) so many GPs
# that were in Ajmer/Jodhpur/Jaipur etc. are now recorded under new
# district names in LGD. Matching on district name therefore fails.
#
# Strategy: match on GP name only, use samiti as tiebreaker for duplicates.

# ── Step 1: Check how many GP names are unique across Rajasthan ──────────────
gp_name_counts = gp_lookup["gp_norm"].value_counts()
print(f"Total unique GP names in LGD (Rajasthan): {gp_lookup['gp_norm'].nunique()}")
print(f"GP names appearing once (unambiguous):    {(gp_name_counts == 1).sum()}")
print(f"GP names appearing 2+ times (ambiguous):  {(gp_name_counts > 1).sum()}")
print(f"\nMost common duplicate GP names:")
print(gp_name_counts[gp_name_counts > 1].head(20))

# ── Step 2: Exact match on GP name only ──────────────────────────────────────
merged_gpname = reservations_clean.merge(
    gp_lookup[["gp_lgd_code", "gp_name_lgd", "gp_norm",
               "district_lgd", "subdistrict_lgd"]],
    left_on="gp_norm_2010",
    right_on="gp_norm",
    how="left"
)

matched   = merged_gpname["gp_lgd_code"].notna().sum()
unmatched = merged_gpname["gp_lgd_code"].isna().sum()
total_unique = reservations_clean["gp_norm_2010"].nunique()
print(f"\n── Match on GP name only ──")
print(f"Total reservation GPs:       {len(reservations_clean)}")
print(f"Matched (including dupes):   {matched}")
print(f"Unmatched:                   {unmatched} ({unmatched/len(reservations_clean):.1%})")

# ── Step 3: Flag ambiguous matches (same GP name → multiple LGD codes) ───────
dupes_gpname = (
    merged_gpname[merged_gpname["gp_lgd_code"].notna()]
    .groupby(["gp_norm_2010", "dist_name_new_2010"])["gp_lgd_code"]
    .nunique()
)
ambiguous = (dupes_gpname > 1).sum()
unique_match = (dupes_gpname == 1).sum()
print(f"\nOf matched GPs:")
print(f"  Unambiguous (1 LGD code):  {unique_match}")
print(f"  Ambiguous (2+ LGD codes):  {ambiguous}")

# ── Step 4: Try samiti → subdistrict to break ties ───────────────────────────
# Normalize samiti names for comparison
reservations_clean["samiti_norm_2010"] = (
    reservations_clean["samiti_name_new_2010"].apply(normalize_name)
)
gp_lookup["subdistrict_norm"] = gp_lookup["subdistrict_lgd"].apply(normalize_name)

print(f"\nSample samiti vs subdistrict names:")
temp = merged_gpname[merged_gpname["gp_lgd_code"].notna()].head(10)
print(temp[["samiti_name_new_2010", "subdistrict_lgd"]].drop_duplicates().head(10))

Total unique GP names in LGD (Rajasthan): 9854
GP names appearing once (unambiguous):    9026
GP names appearing 2+ times (ambiguous):  828

Most common duplicate GP names:
gp_norm
rampura        17
gopalpura      13
kishanpura     11
shyampura      11
amarpura       10
ramgarh        10
rajpura        10
kotri           9
patan           9
gothra          9
kushalpura      9
chainpura       9
govindpura      9
kotra           8
ganeshpura      8
bhagwanpura     7
borda           7
daulatpura      7
ramnagar        7
raipur          7
Name: count, dtype: int64

── Match on GP name only ──
Total reservation GPs:       8290
Matched (including dupes):   7789
Unmatched:                   3579 (43.2%)

Of matched GPs:
  Unambiguous (1 LGD code):  3441
  Ambiguous (2+ LGD codes):  1154

Sample samiti vs subdistrict names:
   samiti_name_new_2010 subdistrict_lgd
0                  ARAI          Sarwar
1                  ARAI          Chaksu
2                  ARAI         Khanpur
5           

In [10]:
# =============================================================================
# CELL 8 — Diagnostic: True Match Rate at Reservation GP Level
# =============================================================================

# Count matches per original reservation GP
match_counts = (
    merged_gpname.groupby(["gp_norm_2010", "dist_name_new_2010"])["gp_lgd_code"]
    .apply(lambda x: x.notna().sum())
    .reset_index()
    .rename(columns={"gp_lgd_code": "n_lgd_matches"})
)

print("Match count distribution per reservation GP:")
print(match_counts["n_lgd_matches"].value_counts().sort_index())

no_match      = (match_counts["n_lgd_matches"] == 0).sum()
unique_match  = (match_counts["n_lgd_matches"] == 1).sum()
multi_match   = (match_counts["n_lgd_matches"] > 1).sum()
total         = len(match_counts)

print(f"\nOf {total} unique reservation GP+district combinations:")
print(f"  No match:          {no_match} ({no_match/total:.1%})")
print(f"  Unique match (1):  {unique_match} ({unique_match/total:.1%})")
print(f"  Multiple matches:  {multi_match} ({multi_match/total:.1%})")

# Check the dist field — might be a numeric district code that helps
print(f"\ndist field sample values:")
print(reservations_clean["dist"].value_counts().head(20))

# Sample no-match cases
no_match_gps = match_counts[match_counts["n_lgd_matches"] == 0]
print(f"\nSample no-match reservation GPs:")
sample = reservations_clean[
    reservations_clean["gp_norm_2010"].isin(no_match_gps["gp_norm_2010"])
][["dist_name_new_2010", "samiti_name_new_2010",
   "gp_new_2010", "gp_norm_2010"]].drop_duplicates().head(20)
print(sample)

Match count distribution per reservation GP:
n_lgd_matches
0     3550
1     3404
2      610
3      193
4      145
5       60
6       55
7       29
8       10
9       22
10      24
11      10
12       4
13       7
14       1
15       1
16       2
17      10
18       5
22       1
30       1
33       1
Name: count, dtype: int64

Of 8145 unique reservation GP+district combinations:
  No match:          3550 (43.6%)
  Unique match (1):  3404 (41.8%)
  Multiple matches:  1191 (14.6%)

dist field sample values:
dist
0    7600
1     409
2     281
Name: count, dtype: int64

Sample no-match reservation GPs:
   dist_name_new_2010 samiti_name_new_2010   gp_new_2010  gp_norm_2010
2               AJMER                 ARAI          arai          arai
4               AJMER                 ARAI      bhamolaw      bhamolaw
9               AJMER                 ARAI       devpuri       devpuri
10              AJMER                 ARAI        dhasuk        dhasuk
18              AJMER                 AR